# 🧠 SAM-AI: Multi-Domain Intelligence Baseline Benchmark (Phase 3A)
### Foundation Model: `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` (Base Only)
---
**Core Evaluation Protocol & Guarantees:**
- **Model:** Untouched `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` (Base weights only).
- **LoRA Adapter:** **STRICTLY NONE** (Zero adapters loaded or attached; Experiment 001 adapter is rejected).
- **Web Retrieval:** **DISABLED** (Evaluates purely intrinsic model reasoning).
- **Decoding:** Greedy deterministic decoding (`temperature=0.0`, `do_sample=False`, `max_new_tokens=2048`).
- **Resumability:** Checkpoint-enabled streaming to `/kaggle/working/predictions.jsonl`.
- **Code Sandbox:** Isolated subprocess environment with 3.0-second execution timeout.
- **Fixtures SHA-256:** `b7fc8795fa64c82a97a871efaee68491fc7aa93f3bb557a31197b710892caf3d` (95 samples across 7 domains).

### Evaluated Domains:
1. `mathematics_gsm8k`: 20 items (Grade school arithmetic reasoning)
2. `mathematics_math500`: 20 items (Challenging competition mathematics)
3. `coding_humaneval`: 20 items (Functional Python problem solving with assertion test suites)
4. `science_reasoning`: 10 items (Physics, chemistry, biology calculations)
5. `logic_constraints`: 10 items (Knights & Knaves, syllogisms, constraint satisfaction)
6. `instruction_following`: 10 items (JSON schema, word limits, letter bans, formatting)
7. `multiturn_context`: 5 dialogue pairs = 10 turns (Multi-turn recall and state tracking)


In [ ]:
# Cell 1: Hardware & Storage Preflight Audit (Fail-Closed)
import os
import sys
import json
import time
import shutil
import platform
import subprocess
from pathlib import Path

print("=" * 70)
print("SAM-AI MULTI-DOMAIN BASELINE: HARDWARE & PREFLIGHT AUDIT")
print("=" * 70)

import torch
cuda_available = torch.cuda.is_available()
print(f"[*] CUDA Available: {cuda_available}")

if not cuda_available:
    raise RuntimeError("Kaggle GPU accelerator is not enabled. Please enable GPU T4 x 2 or P100 in Notebook Settings.")

device_count = torch.cuda.device_count()
device_names = [torch.cuda.get_device_name(i) for i in range(device_count)]
print(f"[*] GPUs detected: {device_count} ({', '.join(device_names)})")

total_vram_gb = 0.0
for i in range(device_count):
    vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    total_vram_gb += vram_gb
    cc = torch.cuda.get_device_capability(i)
    print(f"    - GPU {i} ({device_names[i]}): CC {cc[0]}.{cc[1]} | {vram_gb:.2f} GB total VRAM")

print(f"[*] Total Cluster VRAM: {total_vram_gb:.2f} GB")
if total_vram_gb < 14.0:
    raise RuntimeError(f"Insufficient VRAM for 7B evaluation: detected {total_vram_gb:.2f} GB, minimum required is 14.0 GB.")

working_dir = Path("/kaggle/working" if os.path.exists("/kaggle/working") else "./working")
working_dir.mkdir(parents=True, exist_ok=True)
disk_info = shutil.disk_usage(str(working_dir))
free_disk_gb = disk_info.free / (1024**3)
print(f"[*] Working Directory: {working_dir.resolve()}")
print(f"[*] Free Disk Space: {free_disk_gb:.2f} GB")

if free_disk_gb < 10.0:
    raise RuntimeError(f"Insufficient disk space in working directory: {free_disk_gb:.2f} GB free, minimum 10.0 GB required.")

if os.path.exists("/kaggle/working"):
    print("[*] Kaggle environment detected. Ensuring required packages are installed...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45.0", "accelerate>=0.34.0", "bitsandbytes>=0.43.0"
    ], check=True)

print("[OK] Preflight audit passed successfully.")


In [ ]:
# Cell 2: Benchmark Fixtures & Domain Extraction / Verification Logic
import os
import sys
import re
import json
import hashlib
import tempfile
import subprocess
from pathlib import Path
from fractions import Fraction
from typing import List, Dict, Any, Optional, Tuple

FIXTURES_RAW = '{"version": "1.0.0", "total_domains": 7, "total_samples": 95, "domains": {"mathematics_gsm8k": [{"id": "gsm8k_val_pilot_00", "domain": "mathematics_gsm8k", "question": "Mimi picked up 2 dozen seashells on the beach. Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found. How many seashells did Leigh have?", "target": "16", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_01", "domain": "mathematics_gsm8k", "question": "Frankie\'s parents let him have many pets. He has six more snakes than he has cats. He has one less parrot than cats. Six of his pets have four legs. He has 2 dogs. How many pets does he have in total?", "target": "19", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_02", "domain": "mathematics_gsm8k", "question": "Olaf collects colorful toy cars. At first, his collection consisted of 150 cars. Grandpa gave Olaf twice as many toy cars as the uncle. Dad gave Olaf 10 toy cars, 5 less than Mum. Auntie gave Olaf 6 toy cars, 1 more than the uncle. How many toy cars does Olaf have in total, after receiving all these gifts?", "target": "196", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_03", "domain": "mathematics_gsm8k", "question": "Emma\'s bank account has $100 in it. Each day of the week, she spends $8. At the end of the week, she goes to the bank and asks for as many $5 bills as her account can give her. She leaves the rest in the account. How many dollars remain in the account?", "target": "4", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_04", "domain": "mathematics_gsm8k", "question": "Ezekiel hikes as a hobby. This past summer, he did a challenging three-day hike across 50 kilometers of wilderness. The first day, he covered 10 kilometers of steep mountainside. The second day was flatter and he was able to cover half the full hike distance. How many kilometers did he have to hike on the third day to finish the hike?", "target": "15", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_05", "domain": "mathematics_gsm8k", "question": "James decides to build a tin house by collecting 500 tins in a week. On the first day, he collects 50 tins. On the second day, he manages to collect 3 times that number. On the third day, he collects 50 tins fewer than the number he collected on the second day. If he collects an equal number of tins on the remaining days of the week, what\'s the number of tins he collected each day for the rest of the week?", "target": "50", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_06", "domain": "mathematics_gsm8k", "question": "A farmer has 120 apple trees. Each tree yields 40 apples per season. If he sells the apples in bags of 10 for $5 per bag, how much money does he make in total?", "target": "2400", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_07", "domain": "mathematics_gsm8k", "question": "Sarah bought 4 notebooks for $3 each and 6 pens for $1.50 each. If she handed the cashier a $50 bill, how much change did she receive?", "target": "29", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_08", "domain": "mathematics_gsm8k", "question": "A library has 3 floors. The first floor has 2,500 books. The second floor has 500 fewer books than the first floor. The third floor has twice as many books as the second floor. How many books are in the library altogether?", "target": "8500", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_09", "domain": "mathematics_gsm8k", "question": "Liam drives at 60 miles per hour for 2 hours, then slows down to 45 miles per hour for the next 3 hours. How many total miles did he drive?", "target": "255", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_10", "domain": "mathematics_gsm8k", "question": "A bakery makes 400 cookies every morning. By noon, they have sold 60% of them. In the afternoon, they bake another 150 cookies and sell 80 cookies before closing. How many cookies are left at the end of the day?", "target": "230", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_11", "domain": "mathematics_gsm8k", "question": "Chloe has $45. She wants to buy a pair of shoes that cost $70. If her mother gives her $15 and she earns $6 per hour walking dogs, how many hours must she walk dogs to afford the shoes?", "target": "2", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_12", "domain": "mathematics_gsm8k", "question": "A train travels 360 kilometers in 4 hours. If it increases its speed by 15 kilometers per hour, how far will it travel in the next 3 hours?", "target": "315", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_13", "domain": "mathematics_gsm8k", "question": "Three friends split the cost of a rental cabin equally. The total cost is $360 for 3 nights. One friend leaves after 2 nights, so the remaining two agree to cover the final night equally between themselves. How much does each of the two staying friends pay in total?", "target": "140", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_14", "domain": "mathematics_gsm8k", "question": "A movie theater has 200 seats. For a Friday screening, adults pay $12 and children pay $8. If 150 adults and 50 children attend, what is the total ticket revenue?", "target": "2200", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_15", "domain": "mathematics_gsm8k", "question": "David has a collection of 80 stamps. 25% of the stamps are from Europe, 35% from Asia, and the remaining stamps are from the Americas. How many stamps are from the Americas?", "target": "32", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_16", "domain": "mathematics_gsm8k", "question": "A swimming pool contains 15,000 gallons of water. It is being drained at a rate of 500 gallons per hour. After 8 hours of draining, how many gallons of water remain in the pool?", "target": "11000", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_17", "domain": "mathematics_gsm8k", "question": "An athlete runs 5 kilometers on Monday, twice that on Tuesday, rests on Wednesday, runs 8 kilometers on Thursday, and runs 12 kilometers on Friday. What is his total distance run for the week?", "target": "35", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_18", "domain": "mathematics_gsm8k", "question": "A printer prints 25 black-and-white pages per minute and 10 color pages per minute. If it prints 150 black-and-white pages and 40 color pages, how many total minutes does it take?", "target": "10", "eval_type": "numerical_exact"}, {"id": "gsm8k_val_pilot_19", "domain": "mathematics_gsm8k", "question": "A coffee shop sells 180 cups of coffee on Monday. On Tuesday, sales increase by 20%. On Wednesday, sales decrease by 10% compared to Tuesday. How many cups of coffee were sold on Wednesday?", "target": "194", "eval_type": "numerical_exact"}], "mathematics_math500": [{"id": "math500_01_alg", "domain": "mathematics_math500", "question": "Solve for x: 3x + 7 = 2(x - 4) + 19. What is x?", "target": "4", "eval_type": "numerical_exact"}, {"id": "math500_02_num", "domain": "mathematics_math500", "question": "Find the smallest positive integer n such that 7n is congruent to 1 modulo 23.", "target": "10", "eval_type": "numerical_exact"}, {"id": "math500_03_geom", "domain": "mathematics_math500", "question": "A right triangle has hypotenuse of length 25 and one leg of length 7. What is the area of the triangle?", "target": "84", "eval_type": "numerical_exact"}, {"id": "math500_04_prob", "domain": "mathematics_math500", "question": "How many distinct permutations can be formed using all the letters in the word PROBABILITY?", "target": "9979200", "eval_type": "numerical_exact"}, {"id": "math500_05_precalc", "domain": "mathematics_math500", "question": "If sin(theta) = 3/5 and theta is in the second quadrant, compute tan(theta). Give the exact value.", "target": "-3/4", "eval_type": "math_symbolic"}, {"id": "math500_06_alg", "domain": "mathematics_math500", "question": "If the roots of x^2 - 14x + k = 0 differ by 6, find the value of k.", "target": "40", "eval_type": "numerical_exact"}, {"id": "math500_07_num", "domain": "mathematics_math500", "question": "Compute the sum of all divisors of 72.", "target": "195", "eval_type": "numerical_exact"}, {"id": "math500_08_geom", "domain": "mathematics_math500", "question": "The volume of a sphere is 36*pi cubic centimeters. What is its surface area in square centimeters? State the coefficient of pi.", "target": "36", "eval_type": "numerical_exact"}, {"id": "math500_09_alg", "domain": "mathematics_math500", "question": "Evaluate the infinite geometric series: 12 + 6 + 3 + 1.5 + ...", "target": "24", "eval_type": "numerical_exact"}, {"id": "math500_10_prob", "domain": "mathematics_math500", "question": "Two standard fair 6-sided dice are rolled. What is the probability that the sum of the dice is 9? Give as a reduced fraction.", "target": "1/9", "eval_type": "math_symbolic"}, {"id": "math500_11_num", "domain": "mathematics_math500", "question": "What is the remainder when 2^100 is divided by 7?", "target": "2", "eval_type": "numerical_exact"}, {"id": "math500_12_alg", "domain": "mathematics_math500", "question": "Find the minimum value of f(x) = 2x^2 - 12x + 25.", "target": "7", "eval_type": "numerical_exact"}, {"id": "math500_13_geom", "domain": "mathematics_math500", "question": "A circle has circumference 18*pi. What is the area of a square inscribed inside this circle?", "target": "162", "eval_type": "numerical_exact"}, {"id": "math500_14_num", "domain": "mathematics_math500", "question": "What is the greatest common divisor of 2024 and 748?", "target": "44", "eval_type": "numerical_exact"}, {"id": "math500_15_alg", "domain": "mathematics_math500", "question": "If log_2(x) + log_2(x - 6) = 4, find the value of x.", "target": "8", "eval_type": "numerical_exact"}, {"id": "math500_16_prob", "domain": "mathematics_math500", "question": "In how many ways can a committee of 4 people be chosen from a group of 9 people?", "target": "126", "eval_type": "numerical_exact"}, {"id": "math500_17_precalc", "domain": "mathematics_math500", "question": "Compute the determinant of the matrix [[4, 3], [2, 5]].", "target": "14", "eval_type": "numerical_exact"}, {"id": "math500_18_num", "domain": "mathematics_math500", "question": "How many trailing zeros are at the end of 50! (50 factorial)?", "target": "12", "eval_type": "numerical_exact"}, {"id": "math500_19_alg", "domain": "mathematics_math500", "question": "If a + b = 9 and ab = 18, find a^2 + b^2.", "target": "45", "eval_type": "numerical_exact"}, {"id": "math500_20_geom", "domain": "mathematics_math500", "question": "The coordinates of points A, B, and C are (0,0), (6,0), and (0,8). Find the radius of the circumscribed circle of triangle ABC.", "target": "5", "eval_type": "numerical_exact"}], "coding_humaneval": [{"id": "humaneval_00", "domain": "coding_humaneval", "entry_point": "has_close_elements", "prompt": "from typing import List\\n\\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\\n    \\"\\"\\" Check if in given list of numbers, are any two numbers closer to each other than given threshold.\\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\\n    False\\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\\n    True\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True\\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == False\\n    assert candidate([1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1) == True\\n", "eval_type": "code_execution"}, {"id": "humaneval_01", "domain": "coding_humaneval", "entry_point": "separate_paren_groups", "prompt": "from typing import List\\n\\ndef separate_paren_groups(paren_string: str) -> List[str]:\\n    \\"\\"\\" Input to this function is a string containing multiple groups of nested parentheses. Your goal is to\\n    separate those group into separate strings and return the list of those.\\n    Separate groups are balanced, each group is enclosed in parentheses, and they do not overlap.\\n    Ignore any spaces in the input string.\\n    >>> separate_paren_groups(\'( ) (( )) (( )( ))\')\\n    [\'()\', \'(())\', \'(()())\']\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'(()()) (( )) () ((())()())\') == [\'(()())\', \'(())\', \'()\', \'((())()())\']\\n    assert candidate(\'() (()) ((())) (((())))\') == [\'()\', \'(())\', \'((()))\', \'(((())))\']\\n    assert candidate(\'(()(())((())))\') == [\'(()(())((())))\']\\n    assert candidate(\'( ) (( )) (( )( ))\') == [\'()\', \'(())\', \'(()())\']\\n", "eval_type": "code_execution"}, {"id": "humaneval_02", "domain": "coding_humaneval", "entry_point": "truncate_number", "prompt": "def truncate_number(number: float) -> float:\\n    \\"\\"\\" Given a positive floating point number, it can be decomposed into\\n    and integer part (largest integer smaller than given number) and decimals\\n    (leftover part always smaller than 1).\\n    Return the decimal part of the number.\\n    >>> truncate_number(3.5)\\n    0.5\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert abs(candidate(3.5) - 0.5) < 1e-6\\n    assert abs(candidate(1.33) - 0.33) < 1e-6\\n    assert abs(candidate(123.456) - 0.456) < 1e-6\\n", "eval_type": "code_execution"}, {"id": "humaneval_03", "domain": "coding_humaneval", "entry_point": "below_zero", "prompt": "from typing import List\\n\\ndef below_zero(operations: List[int]) -> bool:\\n    \\"\\"\\" You\'re given a list of deposit and withdrawal operations on a bank account that starts with\\n    zero balance. Your task is to detect if at any point the balance of account fallls below zero, and\\n    at that point function should return True. Otherwise it should return False.\\n    >>> below_zero([1, 2, 3])\\n    False\\n    >>> below_zero([1, 2, -4, 5])\\n    True\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate([]) == False\\n    assert candidate([1, 2, -3, 1, 2, -3]) == False\\n    assert candidate([1, 2, -4, 5, 6]) == True\\n    assert candidate([1, -1, 2, -2, 5, -5, 4, -4]) == False\\n    assert candidate([1, -1, 2, -2, 5, -5, 4, -5]) == True\\n", "eval_type": "code_execution"}, {"id": "humaneval_04", "domain": "coding_humaneval", "entry_point": "mean_absolute_deviation", "prompt": "from typing import List\\n\\ndef mean_absolute_deviation(numbers: List[float]) -> float:\\n    \\"\\"\\" For a given list of input numbers, calculate Mean Absolute Deviation\\n    around the mean of this dataset.\\n    Mean Absolute Deviation is the average absolute difference between each\\n    element and a centerpoint (mean in this case):\\n    MAD = average |x - x_mean|\\n    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])\\n    1.0\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert abs(candidate([1.0, 2.0, 3.0]) - 2.0/3.0) < 1e-6\\n    assert abs(candidate([1.0, 2.0, 3.0, 4.0]) - 1.0) < 1e-6\\n    assert abs(candidate([1.0, 2.0, 3.0, 4.0, 5.0]) - 6.0/5.0) < 1e-6\\n", "eval_type": "code_execution"}, {"id": "humaneval_05", "domain": "coding_humaneval", "entry_point": "intersperse", "prompt": "from typing import List\\n\\ndef intersperse(numbers: List[int], delimeter: int) -> List[int]:\\n    \\"\\"\\" Insert a number \'delimeter\' between every two consecutive elements of input list `numbers\'\\n    >>> intersperse([], 4)\\n    []\\n    >>> intersperse([1, 2, 3], 4)\\n    [1, 4, 2, 4, 3]\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate([], 7) == []\\n    assert candidate([5, 6, 3, 2], 8) == [5, 8, 6, 8, 3, 8, 2]\\n    assert candidate([2, 2, 2], 2) == [2, 2, 2, 2, 2]\\n", "eval_type": "code_execution"}, {"id": "humaneval_06", "domain": "coding_humaneval", "entry_point": "parse_nested_parens", "prompt": "from typing import List\\n\\ndef parse_nested_parens(paren_string: str) -> List[int]:\\n    \\"\\"\\" Input to this function is a string represented multiple groups for nested parentheses separated by spaces.\\n    For each group, output the deepest level of nesting of parentheses.\\n    E.g. (()()) has maximum two levels of nesting while ((())) has three.\\n    >>> parse_nested_parens(\'(()()) ((())) () ((())()())\')\\n    [2, 3, 1, 3]\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'(()()) ((())) () ((())()())\') == [2, 3, 1, 3]\\n    assert candidate(\'() (()) ((())) (((())))\') == [1, 2, 3, 4]\\n    assert candidate(\'(()(())((())))\') == [4]\\n", "eval_type": "code_execution"}, {"id": "humaneval_07", "domain": "coding_humaneval", "entry_point": "filter_by_substring", "prompt": "from typing import List\\n\\ndef filter_by_substring(strings: List[str], substring: str) -> List[str]:\\n    \\"\\"\\" Filter an input list of strings only for ones that contain given substring\\n    >>> filter_by_substring([], \'a\')\\n    []\\n    >>> filter_by_substring([\'abc\', \'bacd\', \'cde\', \'array\'], \'a\')\\n    [\'abc\', \'bacd\', \'array\']\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate([], \'john\') == []\\n    assert candidate([\'xxx\', \'asd\', \'xxy\', \'john\', \'cdd\'], \'xxx\') == [\'xxx\']\\n    assert candidate([\'xxx\', \'asd\', \'aaaxxy\', \'john\', \'cdd\'], \'xx\') == [\'xxx\', \'aaaxxy\']\\n    assert candidate([\'grunt\', \'armory\', \'couch\'], \'arm\') == [\'armory\']\\n", "eval_type": "code_execution"}, {"id": "humaneval_08", "domain": "coding_humaneval", "entry_point": "sum_product", "prompt": "from typing import List, Tuple\\n\\ndef sum_product(numbers: List[int]) -> Tuple[int, int]:\\n    \\"\\"\\" For a given list of integers, return a tuple consisting of a sum and a product of all the integers in a list.\\n    Empty sum should be equal to 0 and empty product should be equal to 1.\\n    >>> sum_product([])\\n    (0, 1)\\n    >>> sum_product([1, 2, 3, 4])\\n    (10, 24)\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate([]) == (0, 1)\\n    assert candidate([1, 1, 1]) == (3, 1)\\n    assert candidate([100, 0]) == (100, 0)\\n    assert candidate([3, 5, 7]) == (3 + 5 + 7, 3 * 5 * 7)\\n    assert candidate([10]) == (10, 10)\\n", "eval_type": "code_execution"}, {"id": "humaneval_09", "domain": "coding_humaneval", "entry_point": "rolling_max", "prompt": "from typing import List, Tuple\\n\\ndef rolling_max(numbers: List[int]) -> List[int]:\\n    \\"\\"\\" From a given list of integers, generate a list of rolling maximum element found until given moment\\n    in the sequence.\\n    >>> rolling_max([1, 2, 3, 2, 3, 4, 2])\\n    [1, 2, 3, 3, 3, 4, 4]\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate([]) == []\\n    assert candidate([1, 2, 3, 4]) == [1, 2, 3, 4]\\n    assert candidate([4, 3, 2, 1]) == [4, 4, 4, 4]\\n    assert candidate([3, 2, 3, 100, 3]) == [3, 3, 3, 100, 100]\\n", "eval_type": "code_execution"}, {"id": "humaneval_10", "domain": "coding_humaneval", "entry_point": "make_palindrome", "prompt": "def is_palindrome(string: str) -> bool:\\n    \\"\\"\\" Test if given string is a palindrome \\"\\"\\"\\n    return string == string[::-1]\\n\\ndef make_palindrome(string: str) -> str:\\n    \\"\\"\\" Find the shortest palindrome that begins with a supplied string.\\n    Algorithm idea is simple:\\n    - Find the longest postfix of supplied string that is a palindrome.\\n    - Append to the end of the string reverse of a string prefix that comes before the palindromic postfix.\\n    >>> make_palindrome(\'\')\\n    \'\'\\n    >>> make_palindrome(\'cat\')\\n    \'catac\'\\n    >>> make_palindrome(\'cata\')\\n    \'catac\'\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'\') == \'\'\\n    assert candidate(\'x\') == \'x\'\\n    assert candidate(\'xyz\') == \'xyzyx\'\\n    assert candidate(\'jerry\') == \'jerryrrej\'\\n", "eval_type": "code_execution"}, {"id": "humaneval_11", "domain": "coding_humaneval", "entry_point": "string_xor", "prompt": "from typing import List\\n\\ndef string_xor(a: str, b: str) -> str:\\n    \\"\\"\\" Input are two strings a and b consisting only of 1s and 0s.\\n    Perform binary XOR on these inputs and return result also as a string.\\n    >>> string_xor(\'010\', \'110\')\\n    \'100\'\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'111000\', \'101010\') == \'010010\'\\n    assert candidate(\'1\', \'1\') == \'0\'\\n    assert candidate(\'0101\', \'0000\') == \'0101\'\\n", "eval_type": "code_execution"}, {"id": "humaneval_12", "domain": "coding_humaneval", "entry_point": "longest", "prompt": "from typing import List, Optional\\n\\ndef longest(strings: List[str]) -> Optional[str]:\\n    \\"\\"\\" Out of list of strings, return the longest one. Return the first one in case of multiple\\n    strings of the same length. Return None in case the input list is empty.\\n    >>> longest([])\\n    None\\n    >>> longest([\'a\', \'b\', \'c\'])\\n    \'a\'\\n    >>> longest([\'a\', \'bb\', \'ccc\'])\\n    \'ccc\'\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate([]) == None\\n    assert candidate([\'x\', \'y\', \'z\']) == \'x\'\\n    assert candidate([\'x\', \'yyy\', \'zzzz\', \'www\', \'kkkk\', \'abc\']) == \'zzzz\'\\n", "eval_type": "code_execution"}, {"id": "humaneval_13", "domain": "coding_humaneval", "entry_point": "greatest_common_divisor", "prompt": "def greatest_common_divisor(a: int, b: int) -> int:\\n    \\"\\"\\" Return a greatest common divisor of two integers a and b\\n    >>> greatest_common_divisor(3, 5)\\n    1\\n    >>> greatest_common_divisor(25, 15)\\n    5\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(3, 7) == 1\\n    assert candidate(10, 15) == 5\\n    assert candidate(49, 14) == 7\\n    assert candidate(144, 60) == 12\\n", "eval_type": "code_execution"}, {"id": "humaneval_14", "domain": "coding_humaneval", "entry_point": "all_prefixes", "prompt": "from typing import List\\n\\ndef all_prefixes(string: str) -> List[str]:\\n    \\"\\"\\" Return list of all prefixes from shortest to longest of the input string\\n    >>> all_prefixes(\'abc\')\\n    [\'a\', \'ab\', \'abc\']\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'\') == []\\n    assert candidate(\'asdfgh\') == [\'a\', \'as\', \'asd\', \'asdf\', \'asdfg\', \'asdfgh\']\\n    assert candidate(\'WWW\') == [\'W\', \'WW\', \'WWW\']\\n", "eval_type": "code_execution"}, {"id": "humaneval_15", "domain": "coding_humaneval", "entry_point": "string_sequence", "prompt": "def string_sequence(n: int) -> str:\\n    \\"\\"\\" Return a string containing space-delimited numbers starting from 0 upto n inclusive.\\n    >>> string_sequence(0)\\n    \'0\'\\n    >>> string_sequence(5)\\n    \'0 1 2 3 4 5\'\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(0) == \'0\'\\n    assert candidate(3) == \'0 1 2 3\'\\n    assert candidate(10) == \'0 1 2 3 4 5 6 7 8 9 10\'\\n", "eval_type": "code_execution"}, {"id": "humaneval_16", "domain": "coding_humaneval", "entry_point": "count_distinct_characters", "prompt": "def count_distinct_characters(string: str) -> int:\\n    \\"\\"\\" Given a string, find out how many distinct characters (case-insensitive) does it consist of\\n    >>> count_distinct_characters(\'xyzXYZ\')\\n    3\\n    >>> count_distinct_characters(\'Jerry\')\\n    4\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'\') == 0\\n    assert candidate(\'abcde\') == 5\\n    assert candidate(\'abcde\' + \'cade\' + \'CADE\') == 5\\n    assert candidate(\'aaaaAAAAaaaa\') == 1\\n    assert candidate(\'Jerry jERRY JeRrY\') == 5\\n", "eval_type": "code_execution"}, {"id": "humaneval_17", "domain": "coding_humaneval", "entry_point": "parse_music", "prompt": "from typing import List\\n\\ndef parse_music(music_string: str) -> List[int]:\\n    \\"\\"\\" Input to this function is a string representing musical notes in a special ASCII format.\\n    Your task is to parse this string and return list of integers corresponding to how many beats does each\\n    not last.\\n    Here is a legend:\\n    \'o\' - whole note, lasts four beats\\n    \'o|\' - half note, lasts two beats\\n    \'.|\' - quater note, lasts one beat\\n    >>> parse_music(\'o o| .| o| o| .| .| .| .| o o\')\\n    [4, 2, 1, 2, 2, 1, 1, 1, 1, 4, 4]\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'\') == []\\n    assert candidate(\'o o o o\') == [4, 4, 4, 4]\\n    assert candidate(\'.| .| .| .|\') == [1, 1, 1, 1]\\n    assert candidate(\'o| o| .| .| o o o|\') == [2, 2, 1, 1, 4, 4, 2]\\n", "eval_type": "code_execution"}, {"id": "humaneval_18", "domain": "coding_humaneval", "entry_point": "how_many_times", "prompt": "def how_many_times(string: str, substring: str) -> int:\\n    \\"\\"\\" Find how many times a given substring can be found in the original string. Count overlaping cases.\\n    >>> how_many_times(\'\', \'a\')\\n    0\\n    >>> how_many_times(\'aaa\', \'a\')\\n    3\\n    >>> how_many_times(\'aaaa\', \'aa\')\\n    3\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'\', \'x\') == 0\\n    assert candidate(\'xyxyxyx\', \'x\') == 4\\n    assert candidate(\'cacacacac\', \'cac\') == 4\\n    assert candidate(\'john doe\', \'john\') == 1\\n", "eval_type": "code_execution"}, {"id": "humaneval_19", "domain": "coding_humaneval", "entry_point": "sort_numbers", "prompt": "from typing import List\\n\\ndef sort_numbers(numbers: str) -> str:\\n    \\"\\"\\" Input is a space-delimited string of numberals from \'zero\' to \'nine\'.\\n    Valid choices are \'zero\', \'one\', \'two\', \'three\', \'four\', \'five\', \'six\', \'seven\', \'eight\' and \'nine\'.\\n    Return the string with numbers sorted from smallest to largest\\n    >>> sort_numbers(\'three one five\')\\n    \'one three five\'\\n    \\"\\"\\"\\n", "test": "def check(candidate):\\n    assert candidate(\'\') == \'\'\\n    assert candidate(\'three\') == \'three\'\\n    assert candidate(\'three five nine\') == \'three five nine\'\\n    assert candidate(\'five zero four seven nine eight\') == \'zero four five seven eight nine\'\\n    assert candidate(\'six five four three two one zero\') == \'zero one two three four five six\'\\n", "eval_type": "code_execution"}], "science_reasoning": [{"id": "sci_01_phys", "domain": "science_reasoning", "question": "An object is dropped from rest from a height of 80 meters. Assuming g = 9.8 m/s^2 is approximated as 10 m/s^2 and neglecting air resistance, how many seconds does it take to reach the ground?", "target": "4", "eval_type": "numerical_exact"}, {"id": "sci_02_chem", "domain": "science_reasoning", "question": "What is the molar mass of water (H2O) in g/mol? (Use H=1.008, O=16.00, round to two decimal places)", "target": "18.02", "eval_type": "numerical_exact"}, {"id": "sci_03_bio", "domain": "science_reasoning", "question": "In humans, how many pairs of chromosomes are present in a normal somatic diploid cell?", "target": "23", "eval_type": "numerical_exact"}, {"id": "sci_04_phys", "domain": "science_reasoning", "question": "A 5 kg block has a force of 20 N applied to it on a frictionless surface. What is the acceleration of the block in m/s^2?", "target": "4", "eval_type": "numerical_exact"}, {"id": "sci_05_chem", "domain": "science_reasoning", "question": "What is the pH of a 0.001 M hydrochloric acid (HCl) solution, assuming complete dissociation?", "target": "3", "eval_type": "numerical_exact"}, {"id": "sci_06_phys", "domain": "science_reasoning", "question": "A circuit has a 12V voltage source and a resistor of 4 ohms. What is the current flowing in amperes according to Ohm\'s law?", "target": "3", "eval_type": "numerical_exact"}, {"id": "sci_07_astro", "domain": "science_reasoning", "question": "Approximately how many minutes does light from the Sun take to reach Earth? Round to the nearest whole integer.", "target": "8", "eval_type": "numerical_exact"}, {"id": "sci_08_chem", "domain": "science_reasoning", "question": "How many electrons are in the outer valence shell of an uncharged neutral Chlorine atom (Z=17)?", "target": "7", "eval_type": "numerical_exact"}, {"id": "sci_09_phys", "domain": "science_reasoning", "question": "What is the kinetic energy in Joules of a 2 kg mass moving at a velocity of 3 m/s? (KE = 0.5 * m * v^2)", "target": "9", "eval_type": "numerical_exact"}, {"id": "sci_10_bio", "domain": "science_reasoning", "question": "In genetics, if two heterozygous parents (Bb) are crossed, what percentage (0 to 100) of their offspring is expected to exhibit the dominant phenotype according to Mendelian genetics?", "target": "75", "eval_type": "numerical_exact"}], "logic_constraints": [{"id": "logic_01", "domain": "logic_constraints", "question": "On the Island of Knights and Knaves, Knights always tell the truth and Knaves always lie. Person A says: \'Both I and B are knaves.\' Person B says nothing. What is Person A (Knight or Knave)?", "target": "Knave", "eval_type": "text_contains"}, {"id": "logic_02", "domain": "logic_constraints", "question": "If all Bloops are Razzies, and all Razzies are Lizzies, are all Bloops definitely Lizzies? Answer Yes or No.", "target": "Yes", "eval_type": "text_contains"}, {"id": "logic_03", "domain": "logic_constraints", "question": "Alice is taller than Bob. Charlie is shorter than Alice but taller than Bob. Who is the shortest among Alice, Bob, and Charlie?", "target": "Bob", "eval_type": "text_contains"}, {"id": "logic_04", "domain": "logic_constraints", "question": "A father and son are in a car accident. The father dies. The boy is rushed to the hospital. The surgeon looks at the boy and says: \'I cannot operate on this boy, he is my son!\' Who is the surgeon?", "target": "mother", "eval_type": "text_contains"}, {"id": "logic_05", "domain": "logic_constraints", "question": "There are five houses in a row. Red is to the immediate left of Blue. Green is to the right of Blue. Yellow is between Red and Orange. If Orange is on the far left, which house is in the middle (3rd from left)?", "target": "Red", "eval_type": "text_contains"}, {"id": "logic_06", "domain": "logic_constraints", "question": "Either the key is in the drawer or it is on the table. The key is not on the table. Where is the key?", "target": "drawer", "eval_type": "text_contains"}, {"id": "logic_07", "domain": "logic_constraints", "question": "If it rains, the street is wet. The street is not wet. Did it rain? Answer Yes or No.", "target": "No", "eval_type": "text_contains"}, {"id": "logic_08", "domain": "logic_constraints", "question": "Five runners (A, B, C, D, E) compete. A finishes before B. C finishes before D. B finishes before C. E finishes after D. Who won first place?", "target": "A", "eval_type": "text_contains"}, {"id": "logic_09", "domain": "logic_constraints", "question": "You have a 3-gallon jug and a 5-gallon jug. Can you measure exactly 4 gallons using these jugs and an unlimited water source? Answer Yes or No.", "target": "Yes", "eval_type": "text_contains"}, {"id": "logic_10", "domain": "logic_constraints", "question": "A box contains 10 red socks and 10 blue socks. In pitch darkness, what is the minimum number of socks you must draw to guarantee having at least one matching pair?", "target": "3", "eval_type": "numerical_exact"}], "instruction_following": [{"id": "ifeval_01_json", "domain": "instruction_following", "question": "Output a valid JSON object with exactly two keys: \'status\' (set to \'success\') and \'code\' (set to 200). Do not include any other commentary.", "target": "json_valid", "eval_type": "json_schema_validation"}, {"id": "ifeval_02_wordcount", "domain": "instruction_following", "question": "Write a sentence describing the sun. Your response must contain exactly 6 words. No more, no less.", "target": "word_count_6", "eval_type": "word_count_exact_6"}, {"id": "ifeval_03_no_e", "domain": "instruction_following", "question": "Write a sentence of at least five words that does NOT contain the letter \'e\' (or \'E\') anywhere.", "target": "no_letter_e", "eval_type": "disallow_letter_e"}, {"id": "ifeval_04_caps", "domain": "instruction_following", "question": "Say \'hello world\' in ALL CAPITAL LETTERS only.", "target": "HELLO WORLD", "eval_type": "text_exact"}, {"id": "ifeval_05_bullet", "domain": "instruction_following", "question": "List three primary colors. Format as a markdown unordered list with exactly three bullet points starting with \'* \'.", "target": "bullet_count_3", "eval_type": "bullet_count_3"}, {"id": "ifeval_06_numbering", "domain": "instruction_following", "question": "Count from 1 to 5. Format each number on its own line preceded by \'Step X:\'.", "target": "step_prefix_5", "eval_type": "step_prefix_5"}, {"id": "ifeval_07_delimiter", "domain": "instruction_following", "question": "Provide the name of the capital of France. Enclose the name strictly inside triple asterisks, like ***Paris***.", "target": "***Paris***", "eval_type": "text_contains"}, {"id": "ifeval_08_comma", "domain": "instruction_following", "question": "List four fruit names separated only by commas with no spaces.", "target": "comma_separated_4", "eval_type": "comma_separated_4"}, {"id": "ifeval_09_lastword", "domain": "instruction_following", "question": "Explain gravity in one sentence. The very last word of your response MUST be \'downward.\'", "target": "ends_with_downward", "eval_type": "ends_with_downward"}, {"id": "ifeval_10_twolines", "domain": "instruction_following", "question": "Write a short poem with exactly two lines separated by a single newline character.", "target": "line_count_2", "eval_type": "line_count_2"}], "multiturn_context": [{"id": "multiturn_01", "domain": "multiturn_context", "turn_1_user": "My favorite fruit is pomegranate and my favorite city is Kyoto.", "turn_1_assistant_prefix": "I have noted that your favorite fruit is pomegranate and your favorite city is Kyoto.", "turn_2_user": "What was the fruit and city I mentioned earlier?", "target": ["pomegranate", "Kyoto"], "eval_type": "contains_all_targets"}, {"id": "multiturn_02", "domain": "multiturn_context", "turn_1_user": "Remember the secret key code is \'OMEGA-447\'. Do not forget it.", "turn_1_assistant_prefix": "Understood. The secret key code OMEGA-447 has been memorized.", "turn_2_user": "Please state the secret key code I told you.", "target": ["OMEGA-447"], "eval_type": "contains_all_targets"}, {"id": "multiturn_03", "domain": "multiturn_context", "turn_1_user": "Let variable x = 45 and variable y = 15.", "turn_1_assistant_prefix": "Recorded x = 45 and y = 15.", "turn_2_user": "What is the difference x - y?", "target": ["30"], "eval_type": "contains_all_targets"}, {"id": "multiturn_04", "domain": "multiturn_context", "turn_1_user": "I am traveling to Norway next Tuesday for 5 days.", "turn_1_assistant_prefix": "Noted your 5-day trip to Norway next Tuesday.", "turn_2_user": "Which country am I visiting and for how many days?", "target": ["Norway", "5"], "eval_type": "contains_all_targets"}, {"id": "multiturn_05", "domain": "multiturn_context", "turn_1_user": "We are designing a robot named Titan that has 6 legs.", "turn_1_assistant_prefix": "Noted: Robot Titan with 6 legs.", "turn_2_user": "How many legs does our robot have and what is its name?", "target": ["Titan", "6"], "eval_type": "contains_all_targets"}]}}'
BENCHMARK_FIXTURES = json.loads(FIXTURES_RAW)

calculated_sha = hashlib.sha256(FIXTURES_RAW.encode("utf-8")).hexdigest()
print(f"[*] Benchmark Fixtures Loaded: {BENCHMARK_FIXTURES['total_samples']} samples across {BENCHMARK_FIXTURES['total_domains']} domains.")
print(f"[*] Fixtures Integrity Check: SHA-256 = {calculated_sha[:16]}...")

def clean_numeric_str(raw: str) -> Optional[str]:
    """Cleans a raw numeric string, stripping currencies, commas, and trailing punctuation."""
    if not raw:
        return None
    c = raw.replace("$", "").replace(",", "").strip().rstrip(".")
    if c.startswith("+"):
        c = c[1:]
    return c if c else None


def extract_boxed_content(text: str) -> List[str]:
    """Finds all \\boxed{...} instances, properly handling nested curly braces."""
    results = []
    idx = 0
    pattern = "\\boxed{"
    while True:
        pos = text.find(pattern, idx)
        if pos == -1:
            break
        start = pos + len(pattern)
        depth = 1
        curr = start
        while curr < len(text) and depth > 0:
            if text[curr] == "{":
                depth += 1
            elif text[curr] == "}":
                depth -= 1
            curr += 1
        if depth == 0:
            results.append(text[start:curr-1].strip())
        idx = curr
    return results


def extract_numerical_answer(text: str) -> Tuple[Optional[str], str]:
    """
    Extracts numerical answer from text using prioritized regex patterns:
    1. #### <num>
    2. \\boxed{<num>}
    3. Natural phrases (e.g. 'the answer is <num>')
    4. Trailing numeric token in the post-thinking body
    """
    if not text:
        return None, "empty_text"

    # Remove reasoning chain if present
    body = text.split("</think>")[-1] if "</think>" in text else text

    # 1. #### marker (GSM8K canonical)
    hm = re.search(r"####\s*([+-]?(?:\d+[\d,]*\.?\d*|\.\d+))", text)
    if hm:
        v = clean_numeric_str(hm.group(1))
        if v:
            return v, "hash_marker"

    # 2. LaTeX boxed marker
    boxed_candidates = extract_boxed_content(text)
    if boxed_candidates:
        cand = boxed_candidates[-1].strip()
        nc = re.sub(r"[^\d\.\+\-]", "", cand)
        v = clean_numeric_str(nc)
        if v:
            return v, "latex_boxed"

    # 3. Natural language phrase
    pm = re.findall(
        r"(?:the\s+final\s+answer\s+is|the\s+answer\s+is|answer\s*[:=])\s*([+-]?\$?\d+[\d,]*\.?\d*)",
        body,
        re.IGNORECASE
    )
    if pm:
        v = clean_numeric_str(pm[-1])
        if v:
            return v, "natural_phrase"

    # 4. Trailing numeric token fallback
    nums = re.findall(r"(?<!\w)([+-]?\d+[\d,]*\.?\d*)(?!\w)", body)
    if nums:
        v = clean_numeric_str(nums[-1])
        if v:
            return v, "trailing_numeric_fallback"

    return None, "no_number_found"


def extract_symbolic_math(text: str) -> Tuple[Optional[str], str]:
    """
    Extracts symbolic or fractional mathematical answers (e.g., '-3/4', '1/9', 'pi', 'sqrt(2)').
    """
    if not text:
        return None, "empty_text"

    # 1. LaTeX boxed (with nested brace support)
    boxed_candidates = extract_boxed_content(text)
    if boxed_candidates:
        cand = boxed_candidates[-1].strip()
        # Handle latex frac/dfrac: e.g. -\frac{3}{4}, \dfrac{1}{9}, or \frac{-3}{4}
        frac_match = re.search(r"([+-]?)\s*\\(?:d)?frac\{([+-]?\d+)\}\{([+-]?\d+)\}", cand)
        if frac_match:
            prefix_sign = frac_match.group(1)
            num_val = int(frac_match.group(2))
            denom_val = int(frac_match.group(3))
            if prefix_sign == "-":
                num_val = -num_val
            cand = f"{num_val}/{denom_val}"
        cand = cand.replace(" ", "").replace("$", "")
        return cand, "latex_boxed_symbolic"

    # 2. Hash marker
    hm = re.search(r"####\s*([^\r\n]+)", text)
    if hm:
        cand = hm.group(1).strip().replace(" ", "").replace("$", "")
        return cand, "hash_marker_symbolic"

    # 3. Fractions like -3/4 or 1/9
    body = text.split("</think>")[-1] if "</think>" in text else text
    frac_matches = re.findall(r"([+-]?\d+\s*/\s*\d+)", body)
    if frac_matches:
        cand = frac_matches[-1].replace(" ", "")
        return cand, "fraction_match"

    # Fall back to general numerical extraction
    return extract_numerical_answer(text)


def compare_math_answers(pred: Optional[str], target: str, eval_type: str) -> bool:
    """Compares predicted math answer against target with tolerance and fractional equivalence."""
    if pred is None:
        return False
    pred = pred.strip()
    target = target.strip()

    # Exact string match
    if pred.lower() == target.lower():
        return True

    # Fractional equivalence: e.g. -3/4 == -0.75 or 2/4 == 1/2
    try:
        f_pred = Fraction(pred)
        f_target = Fraction(target)
        if f_pred == f_target:
            return True
    except Exception:
        pass

    # Float comparison with tolerance
    try:
        fl_pred = float(pred)
        fl_target = float(target)
        if abs(fl_pred - fl_target) < 1e-5:
            return True
    except Exception:
        pass

    return False


def extract_code_block(completion: str, entry_point: Optional[str] = None) -> str:
    """
    Extracts Python code from model completion.
    Strips reasoning tags (<think>...</think>) and pulls code from markdown blocks
    or lines defining the entry_point.
    """
    if not completion:
        return ""

    body = completion.split("</think>")[-1] if "</think>" in completion else completion

    # Match ```python ... ```
    code_blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", body, re.DOTALL)
    if code_blocks:
        if entry_point:
            for block in reversed(code_blocks):
                if f"def {entry_point}" in block:
                    return block.strip("` \t\r\n")
        return code_blocks[-1].strip("` \t\r\n")

    # If no markdown fences, check if there's raw python code
    lines = body.split("\n")
    code_lines = []
    started = False
    for line in lines:
        if entry_point and f"def {entry_point}" in line:
            started = True
        if started or line.startswith("def ") or line.startswith("import ") or line.startswith("from "):
            started = True
            code_lines.append(line)

    if code_lines:
        return "\n".join(code_lines).strip("` \t\r\n")

    return body.strip("` \t\r\n")


def execute_humaneval_in_sandbox(
    prompt: str,
    completion: str,
    test_code: str,
    entry_point: str,
    timeout_seconds: float = 3.0
) -> Tuple[bool, str, str]:
    """
    Executes HumanEval candidate solution within an isolated subprocess sandbox.
    Returns (passed, details, extracted_code).
    """
    code_body = extract_code_block(completion, entry_point)
    if not code_body:
        return False, "no_code_extracted", ""

    # If code_body already defines entry_point, do not duplicate prompt header
    if entry_point and f"def {entry_point}" in code_body:
        full_code = code_body
    else:
        full_code = prompt.rstrip() + "\n" + code_body

    full_script = f"""
# HumanEval Isolated Execution Sandbox
import sys
import math
from typing import List, Tuple, Optional, Dict, Any, Union

{full_code}

{test_code}

try:
    check({entry_point})
    sys.exit(0)
except AssertionError as e:
    sys.stderr.write(f"AssertionError: {{e}}\\n")
    sys.exit(1)
except Exception as e:
    sys.stderr.write(f"RuntimeError: {{type(e).__name__}}: {{e}}\\n")
    sys.exit(2)
"""

    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as tf:
        tf.write(full_script)
        temp_path = tf.name

    try:
        res = subprocess.run(
            [sys.executable, temp_path],
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            env={
                "PATH": os.environ.get("PATH", ""),
                "PYTHONPATH": "",
                "HF_HUB_OFFLINE": "1",
                "TRANSFORMERS_OFFLINE": "1"
            }
        )
        if res.returncode == 0:
            return True, "passed_all_tests", code_body
        else:
            err = res.stderr.strip().split("\n")[-1] if res.stderr else f"exit_code_{res.returncode}"
            return False, f"test_failed: {err}", code_body
    except subprocess.TimeoutExpired:
        return False, f"timeout_exceeded_{timeout_seconds}s", code_body
    except Exception as e:
        return False, f"execution_error: {e}", code_body
    finally:
        try:
            os.remove(temp_path)
        except OSError:
            pass


def verify_instruction_constraint(eval_type: str, text: str) -> Tuple[bool, str]:
    """Verifies strict instruction-following constraints."""
    if not text:
        return False, "empty_text"

    body = text.split("</think>")[-1].strip() if "</think>" in text else text.strip()

    if eval_type == "json_schema_validation":
        # Look for JSON object with status='success' and code=200
        json_matches = re.findall(r"\{.*?\}", body, re.DOTALL)
        for m in json_matches:
            try:
                data = json.loads(m)
                if data.get("status") == "success" and data.get("code") == 200:
                    return True, "valid_json_matching_schema"
            except Exception:
                continue
        return False, "no_valid_json_matching_schema"

    elif eval_type == "word_count_exact_6":
        words = [w for w in re.split(r"\s+", body) if w]
        if len(words) == 6:
            return True, "exact_6_words"
        return False, f"word_count_is_{len(words)}"

    elif eval_type == "disallow_letter_e":
        # Check if the letter e or E appears anywhere
        words = [w for w in re.split(r"\s+", body) if w]
        if len(words) < 5:
            return False, f"insufficient_words_{len(words)}"
        if "e" in body.lower():
            return False, "contains_letter_e"
        return True, "no_letter_e_found"

    elif eval_type == "text_exact":
        # Check if text is uppercase "HELLO WORLD"
        if "HELLO WORLD" in body:
            return True, "text_exact_match"
        return False, f"mismatch_{body[:30]}"

    elif eval_type == "bullet_count_3":
        bullets = [line.strip() for line in body.split("\n") if line.strip().startswith("* ")]
        if len(bullets) == 3:
            return True, "exact_3_bullets"
        return False, f"bullet_count_is_{len(bullets)}"

    elif eval_type == "step_prefix_5":
        steps = [line.strip() for line in body.split("\n") if re.match(r"^Step\s+[1-5]:", line.strip(), re.IGNORECASE)]
        if len(steps) >= 5:
            return True, "has_5_steps"
        return False, f"step_count_is_{len(steps)}"

    elif eval_type == "text_contains":
        return True, "contains_target"

    elif eval_type == "comma_separated_4":
        # Look for 4 items separated only by commas with no spaces
        lines = [line.strip() for line in body.split("\n") if line.strip()]
        for line in lines:
            parts = line.split(",")
            if len(parts) == 4 and all(len(p) > 0 and " " not in p for p in parts):
                return True, "valid_comma_separated_4"
        return False, "invalid_comma_separation"

    elif eval_type == "ends_with_downward":
        clean = body.rstrip(". \t\n\r")
        if clean.lower().endswith("downward"):
            return True, "ends_with_downward"
        return False, f"trailing_word_is_{clean.split()[-1] if clean.split() else 'none'}"

    elif eval_type == "line_count_2":
        lines = [l for l in body.split("\n") if l.strip()]
        if len(lines) == 2:
            return True, "exact_2_lines"
        return False, f"line_count_is_{len(lines)}"

    return False, f"unsupported_eval_type_{eval_type}"


def verify_multiturn_context(text: str, targets: List[str]) -> Tuple[bool, str]:
    """Verifies that the multi-turn answer recalls all required context items."""
    if not text:
        return False, "empty_text"
    body = text.split("</think>")[-1].lower() if "</think>" in text else text.lower()
    missing = []
    for tgt in targets:
        if tgt.lower() not in body:
            missing.append(tgt)

    if not missing:
        return True, "all_context_targets_recalled"
    return False, f"missing_context_targets: {missing}"


def verify_prediction(
    domain: str,
    eval_type: str,
    raw_output: str,
    target: Any,
    test_suite: Optional[str] = None,
    entry_point: Optional[str] = None,
    prompt: Optional[str] = None
) -> Tuple[bool, Optional[str], str]:
    """
    Master dispatcher verifying a single prediction across any of the 7 benchmark domains.
    Returns: (is_correct, extracted_value, verification_method)
    """
    if domain in ("mathematics_gsm8k", "mathematics_math500"):
        if eval_type == "numerical_exact":
            pred, method = extract_numerical_answer(raw_output)
            is_corr = compare_math_answers(pred, str(target), eval_type)
            return is_corr, pred, method
        elif eval_type == "math_symbolic":
            pred, method = extract_symbolic_math(raw_output)
            is_corr = compare_math_answers(pred, str(target), eval_type)
            return is_corr, pred, method

    elif domain == "coding_humaneval":
        passed, details, extracted_code = execute_humaneval_in_sandbox(
            prompt=prompt or "",
            completion=raw_output,
            test_code=test_suite or "",
            entry_point=entry_point or "check",
            timeout_seconds=3.0
        )
        return passed, extracted_code[:100] if extracted_code else None, details

    elif domain in ("science_reasoning", "logic_constraints"):
        if eval_type == "numerical_exact":
            pred, method = extract_numerical_answer(raw_output)
            is_corr = compare_math_answers(pred, str(target), eval_type)
            return is_corr, pred, method
        elif eval_type == "text_contains":
            body = raw_output.split("</think>")[-1].lower() if "</think>" in raw_output else raw_output.lower()
            tgt_clean = str(target).lower().strip()
            is_corr = tgt_clean in body
            return is_corr, tgt_clean if is_corr else None, "substring_search"

    elif domain == "instruction_following":
        passed, details = verify_instruction_constraint(eval_type, raw_output)
        return passed, details, eval_type

    elif domain == "multiturn_context":
        targets = target if isinstance(target, list) else [str(target)]
        passed, details = verify_multiturn_context(raw_output, targets)
        return passed, ", ".join(targets), details

    return False, None, "unrecognized_domain"
print("[OK] Benchmark verifiers and sandbox engine successfully initialized.")


In [ ]:
# Cell 3: Fail-Closed Base Model Loader (deepseek-ai/DeepSeek-R1-Distill-Qwen-7B)
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

print("=" * 70)
print(f"LOADING FOUNDATION MODEL: {MODEL_ID}")
print("Strict Policy: BASE MODEL ONLY. NO ADAPTER.")
print("=" * 70)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

max_mem = {i: "13.0GB" for i in range(device_count)}

try:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory=max_mem,
        trust_remote_code=True
    )
    print("[*] Successfully loaded base model with 4-bit NF4 BitsAndBytesConfig.")
except Exception as bnb_err:
    print(f"[!] 4-bit NF4 not loaded ({bnb_err}). Loading with float16 sharded across GPUs...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=max_mem,
        trust_remote_code=True
    )
    print("[*] Successfully loaded base model in native float16.")

base_model.eval()

# Strict verification that NO PEFT or LoRA adapter is attached
if hasattr(base_model, "peft_config"):
    raise RuntimeError("CRITICAL ERROR: A PEFT adapter was detected on the base model! Benchmark requires untouched base model.")
if "PeftModel" in type(base_model).__name__:
    raise RuntimeError("CRITICAL ERROR: Model is wrapped in PeftModel! Benchmark requires untouched base model.")

param_count = sum(p.numel() for p in base_model.parameters())
load_dur = time.time() - t0
peak_vram_mb = torch.cuda.max_memory_allocated() / (1024**2)
print(f"[OK] Base model verified untouched. Total parameters: {param_count:,}")
print(f"[OK] Loaded in {load_dur:.2f}s | Peak VRAM: {peak_vram_mb:.2f} MB")


In [ ]:
# Cell 4: Resumable Multi-Domain Execution Loop
checkpoint_file = working_dir / "predictions.jsonl"
print(f"[*] Evaluation Checkpoint File: {checkpoint_file.resolve()}")

completed_ids = set()
if checkpoint_file.exists():
    with open(checkpoint_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    rec = json.loads(line)
                    completed_ids.add(rec["example_id"])
                except Exception:
                    pass
    print(f"[*] Resuming from existing checkpoint: {len(completed_ids)} already completed items found.")
else:
    print("[*] No existing checkpoint found. Starting fresh evaluation.")

# Flatten all 95 samples in deterministic domain order
all_samples = []
for domain_name, items in BENCHMARK_FIXTURES["domains"].items():
    for item in items:
        all_samples.append(item)

print(f"[*] Total items to evaluate: {len(all_samples)}")

bench_start_time = time.time()
ckpt_writer = open(checkpoint_file, "a", encoding="utf-8")

for idx, sample in enumerate(all_samples):
    sid = sample["id"]
    domain = sample["domain"]
    eval_type = sample.get("eval_type", "numerical_exact")
    target = sample.get("target")

    if sid in completed_ids:
        print(f"[{idx+1:02d}/{len(all_samples)}] [RESUMED] Domain: {domain:<22} | ID: {sid}")
        continue

    # Format Prompt based on domain
    if domain in ("mathematics_gsm8k", "mathematics_math500"):
        prompt_text = f"<｜User｜>{sample['question']}\nPlease reason step-by-step and end your response with #### followed by the final numeric answer.\n<｜Assistant｜><think>\n"
    elif domain == "coding_humaneval":
        prompt_text = f"<｜User｜>{sample['prompt']}\nComplete the Python function above. Provide valid executable Python code.\n<｜Assistant｜><think>\n"
    elif domain in ("science_reasoning", "logic_constraints"):
        prompt_text = f"<｜User｜>{sample['question']}\nPlease reason step-by-step and state your final answer clearly at the end.\n<｜Assistant｜><think>\n"
    elif domain == "instruction_following":
        prompt_text = f"<｜User｜>{sample['question']}\nFollow all instructions strictly.\n<｜Assistant｜><think>\n"
    elif domain == "multiturn_context":
        t1_u = sample["turn_1_user"]
        t1_a = sample["turn_1_assistant_prefix"]
        t2_u = sample["turn_2_user"]
        prompt_text = f"<｜User｜>{t1_u}\n<｜Assistant｜>{t1_a} </s>\n<｜User｜>{t2_u}\n<｜Assistant｜><think>\n"
    else:
        prompt_text = f"<｜User｜>{sample.get('question', '')}\n<｜Assistant｜><think>\n"

    inputs = tokenizer(prompt_text, return_tensors="pt").to(base_model.device)
    prompt_len = inputs["input_ids"].shape[1]

    t_start = time.time()
    with torch.no_grad():
        output = base_model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id
        )
    dur_ms = (time.time() - t_start) * 1000.0

    gen_ids = output[0][prompt_len:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    is_truncated = (len(gen_ids) >= 2048)

    # Dispatch universal verification
    is_corr, pred, method = verify_prediction(
        domain=domain,
        eval_type=eval_type,
        raw_output=gen_text,
        target=target,
        test_suite=sample.get("test"),
        entry_point=sample.get("entry_point"),
        prompt=sample.get("prompt")
    )

    rec = {
        "example_id": sid,
        "domain": domain,
        "eval_type": eval_type,
        "target": target,
        "prompt": prompt_text,
        "raw_output": gen_text,
        "extracted_value": pred,
        "verification_method": method,
        "is_correct": is_corr,
        "is_truncated": is_truncated,
        "generated_tokens": len(gen_ids),
        "duration_ms": round(dur_ms, 2)
    }

    # Persist immediately (flush to disk)
    ckpt_writer.write(json.dumps(rec) + "\n")
    ckpt_writer.flush()
    completed_ids.add(sid)

    status_tag = "PASS" if is_corr else "FAIL"
    print(f"[{idx+1:02d}/{len(all_samples)}] [{status_tag}] Domain: {domain:<22} | ID: {sid} | Pred: {str(pred)[:20]} | Toks: {len(gen_ids)} | {dur_ms:.0f}ms")

ckpt_writer.close()
total_eval_time = time.time() - bench_start_time
print()
print(f"[OK] Evaluation execution completed in {total_eval_time:.1f}s.")


In [ ]:
# Cell 5: Aggregation, Summary Report & Packaging
import zipfile

print("=" * 70)
print("SAM-AI MULTI-DOMAIN BASELINE: AGGREGATE RESULTS SUMMARY")
print("=" * 70)

all_records = []
with open(checkpoint_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            all_records.append(json.loads(line))

# Domain Breakdown
domain_stats = {}
for rec in all_records:
    dom = rec["domain"]
    if dom not in domain_stats:
        domain_stats[dom] = {
            "total": 0,
            "correct": 0,
            "total_ms": 0.0,
            "total_tokens": 0,
            "truncated": 0
        }
    domain_stats[dom]["total"] += 1
    if rec["is_correct"]:
        domain_stats[dom]["correct"] += 1
    domain_stats[dom]["total_ms"] += rec["duration_ms"]
    domain_stats[dom]["total_tokens"] += rec["generated_tokens"]
    if rec.get("is_truncated", False):
        domain_stats[dom]["truncated"] += 1

total_correct = sum(s["correct"] for s in domain_stats.values())
total_items = len(all_records)
overall_acc = (total_correct / total_items * 100.0) if total_items > 0 else 0.0

# Print clean ASCII table
print(f"{'Domain':<26} | {'Samples':<8} | {'Correct':<8} | {'Accuracy':<10} | {'Avg Latency':<12} | {'Avg Tokens':<10}")
print("-" * 85)
for dom, stats in domain_stats.items():
    dom_acc = (stats["correct"] / stats["total"] * 100.0) if stats["total"] > 0 else 0.0
    avg_lat = stats["total_ms"] / stats["total"] if stats["total"] > 0 else 0.0
    avg_tok = stats["total_tokens"] / stats["total"] if stats["total"] > 0 else 0.0
    print(f"{dom:<26} | {stats['total']:<8} | {stats['correct']:<8} | {dom_acc:6.2f}%   | {avg_lat:8.1f} ms  | {avg_tok:8.1f}")
print("-" * 85)
print(f"{'OVERALL BASELINE':<26} | {total_items:<8} | {total_correct:<8} | {overall_acc:6.2f}%   |")
print("=" * 85)

# Generate JSON Report
report = {
    "benchmark_title": "SAM-AI Multi-Domain Intelligence Baseline",
    "foundation_model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "adapter_attached": None,
    "total_samples": total_items,
    "overall_correct": total_correct,
    "overall_accuracy_percent": round(overall_acc, 2),
    "total_duration_seconds": round(total_eval_time, 2),
    "fixtures_sha256": calculated_sha,
    "domain_breakdown": {
        dom: {
            "total_samples": stats["total"],
            "correct_count": stats["correct"],
            "accuracy_percent": round((stats["correct"] / stats["total"] * 100.0), 2) if stats["total"] > 0 else 0.0,
            "avg_latency_ms": round(stats["total_ms"] / stats["total"], 2) if stats["total"] > 0 else 0.0,
            "avg_tokens": round(stats["total_tokens"] / stats["total"], 2) if stats["total"] > 0 else 0.0,
            "truncation_count": stats["truncated"]
        }
        for dom, stats in domain_stats.items()
    },
    "hardware": {
        "device_count": device_count,
        "device_names": device_names,
        "peak_vram_mb": round(torch.cuda.max_memory_allocated() / (1024**2), 2)
    }
}

report_file = working_dir / "baseline_multidomain_report.json"
with open(report_file, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print(f"[OK] Report generated: {report_file.resolve()}")

# Package Results into ZIP archive
zip_path = working_dir / "sam_ai_baseline_multidomain_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(report_file, arcname="baseline_multidomain_report.json")
    zf.write(checkpoint_file, arcname="predictions.jsonl")
print(f"[OK] Packaged results archive: {zip_path.resolve()} ({zip_path.stat().st_size / 1024:.1f} KB)")
